In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from upsetplot import plot as upset_plot

from climate_attitudes.settings import Config, RawDataFile
from climate_attitudes.visualisation import configure_mpl

configure_mpl(Path("../fonts/"))

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(156)

config = Config(_env_file="../.env")

In [ ]:
data_w1_to_w5 = (
    pl.read_parquet(RawDataFile.Waves1to5Responses.filepath(config))
    .filter(pl.col("PID").is_not_null())
    .with_columns(
        pl.col("WAVE").cast(int).alias("wave"),
        pl.col("PID").cast(int).alias("participant_id"),
    )
    .select("wave", "participant_id")
)

data_w6 = (
    pl.read_parquet(RawDataFile.Wave6Responses.filepath(config))
    .filter(pl.col("PID").is_not_null())
    .with_columns(
        pl.col("WAVE").cast(int).alias("wave"),
        pl.col("PID").cast(int).alias("participant_id"),
    )
    .select("wave", "participant_id")
)

data = pl.concat((data_w1_to_w5, data_w6)).with_columns(
    pl.col("wave").replace_strict({i: f"Wave {i}" for i in range(1, 7)})
)

In [ ]:
indicators = (
    data.with_columns(pl.lit(True).alias("is_present"))
    .sort(by="wave")
    .pivot(on="wave", index="participant_id")
    .fill_null(False)
)
wave_cols = [c for c in indicators.columns if c != "participant_id"]

rows = []
for r in range(1, len(wave_cols) + 1):
    for combo in combinations(wave_cols, r):
        count = indicators.filter(pl.all_horizontal([pl.col(c) for c in combo])).height
        rows.append({**{c: (c in combo) for c in wave_cols}, "count": count})

plot_data = pl.DataFrame(rows).to_pandas().set_index(wave_cols)["count"]

In [ ]:
fig = plt.figure(figsize=(4.5, 3.5), constrained_layout=True)

plot_result = upset_plot(
    plot_data,
    sort_by="cardinality",
    sort_categories_by="-input",
    min_subset_size=1500,
    totals_plot_elements=0,
    fig=fig,
    element_size=30,
)

plot_result["intersections"].set_ylabel("Response Count")